# **SUMMARIZATION EVALUATION**

## **Imports**

In [1]:
import pandas as pd
from rouge_score import rouge_scorer
from tqdm import tqdm
from bert_score import score
import json
import os

e:\Final Year Project\Speech-to-Text Summarization System for Smart Note-Taking\Virtual\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## **ENGLISH**

### **1. BART English Model Evaluation**

In [2]:
file_path = "../experiments/summarization/bart_samsum/predictions.csv"

df_bart_english = pd.read_csv(file_path)

df_bart_english.head()

,dialogue,reference_summary,generated_summary
0,"Hannah: Hey, do you have Betty's number?\nAman...",Hannah needs Betty's number but Amanda doesn't...,anda can't find Betty's number. she's looking ...
1,Eric: MACHINE!\nRob: That's so gr8!\nEric: I k...,Eric and Rob are going to watch a stand-up on ...,rachel shows how Americans see Russian. they l...
2,"Lenny: Babe, can you help me with something?\n...",Lenny can't decide which trousers to buy. Bob ...,lenny will buy the first or the third pair of ...
3,"Will: hey babe, what do you want for dinner to...",Emma will be home soon and she will let Will k...,will will pick up Emma and will tell her when ...
4,"Ollie: Hi , are you in Warsaw\nJane: yes, just...",Jane is in Warsaw. Ollie and Jane has a party....,jane and ollie are in Warsaw. they will have l...


In [3]:
# Clean up text (if needed)
def CleanText_english(df):
    df = df.dropna()

    df = df[
        (df["reference_summary"].str.strip() != "") &
        (df["generated_summary"].str.strip() != "")
    ]

    print("Total samples:", len(df))
    return df
df_bart_english = CleanText_english(df_bart_english)

Total samples: 819


**ROUGE Evaluation**

In [4]:
# Rouge evaluation
def evaluate_rouge_english(df):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

    rouge1, rouge2, rougel = [], [], []

    for ref, pred in tqdm(zip(df["reference_summary"], df["generated_summary"]), total=len(df)):
        scores = scorer.score(ref, pred)
        
        rouge1.append(scores["rouge1"].fmeasure)
        rouge2.append(scores["rouge2"].fmeasure)
        rougel.append(scores["rougeL"].fmeasure)

    print("🔹 ROUGE Scores:")
    print("ROUGE-1:", sum(rouge1)/len(rouge1))
    print("ROUGE-2:", sum(rouge2)/len(rouge2))
    print("ROUGE-L:", sum(rougel)/len(rougel))
    
    return rouge1, rouge2, rougel
    
rouge1, rouge2, rougel = evaluate_rouge_english(df_bart_english)

  0%|          | 0/819 [00:00<?, ?it/s]

100%|██████████| 819/819 [00:00<00:00, 2058.49it/s]

🔹 ROUGE Scores:
ROUGE-1: 0.40634505108157704
ROUGE-2: 0.18596156954803703
ROUGE-L: 0.32054615354452287


**BERTScore Evaluation**

In [5]:
# BERTScore evaluation
def evaluate_bertscore_english(df):
    preds = df["generated_summary"].tolist()
    refs = df["reference_summary"].tolist()

    P, R, F1 = score(preds, refs, lang="en")

    print("\n🔹 BERTScore:")
    print("Precision:", P.mean().item())
    print("Recall:", R.mean().item())
    print("F1:", F1.mean().item())
    
    return P, R, F1

P, R, F1 = evaluate_bertscore_english(df_bart_english)

e:\Final Year Project\Speech-to-Text Summarization System for Smart Note-Taking\Virtual\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



🔹 BERTScore:
Precision: 0.8892807364463806
Recall: 0.9051057696342468
F1: 0.8969486355781555


**Save Results**

In [6]:
def save_results(df, rouge_results, bertscore_results):
    rouge1, rouge2, rougel = rouge_results
    P, R, F1 = bertscore_results

    results = {
    "ROUGE-1": sum(rouge1)/len(rouge1),
    "ROUGE-2": sum(rouge2)/len(rouge2),
    "ROUGE-L": sum(rougel)/len(rougel),
    "Precision": P.mean().item(),
    "Recall": R.mean().item(),
    "BERTScore-F1": F1.mean().item()
    }

    os.makedirs(output_dir, exist_ok=True)
    with open(f"{output_dir}/results.json", "w") as f:
        json.dump(results, f, indent=4)

    print("✅ Results saved")

output_dir = "../evaluation/summarization/bart_samsum"
rouge_results = (rouge1, rouge2, rougel)
bertscore_results = (P, R, F1)
save_results(df_bart_english, rouge_results, bertscore_results)

✅ Results saved


*Obervation:*

- ROUGE ~0.40 → good content coverage
- ROUGE-2 lower → expected (abstractive model)
- BERTScore ~0.89 → very strong semantic understanding
- Precision + Recall balanced → not hallucinating, not missing info
- However model fails to get  the context of the conversation.

### **2. FlanT5**

In [7]:
file_path = "../experiments/summarization/flanT5_samsum/predictions.csv"

df_bart_english = pd.read_csv(file_path)

print(df_bart_english.head())

P, R, F1 = evaluate_bertscore_english(df_bart_english)
rouge1, rouge2, rougel = evaluate_rouge_english(df_bart_english)
output_dir = "../evaluation/summarization/FlanT5_samsum"
rouge_results = (rouge1, rouge2, rougel)
bertscore_results = (P, R, F1)
save_results(df_bart_english, rouge_results, bertscore_results)

                                            dialogue  \
0  Hannah: Hey, do you have Betty's number?\nAman...   
1  Eric: MACHINE!\nRob: That's so gr8!\nEric: I k...   
2  Lenny: Babe, can you help me with something?\n...   
3  Will: hey babe, what do you want for dinner to...   
4  Ollie: Hi , are you in Warsaw\nJane: yes, just...   

                                   reference_summary  \
0  Hannah needs Betty's number but Amanda doesn't...   
1  Eric and Rob are going to watch a stand-up on ...   
2  Lenny can't decide which trousers to buy. Bob ...   
3  Emma will be home soon and she will let Will k...   
4  Jane is in Warsaw. Ollie and Jane has a party....   

                                   generated_summary  
0  Amanda can't find Betty's number. Amanda asks ...  
1     Eric and Rob like Eric's stand-ups on youtube.  
2  Lenny wants to buy two pairs of purple trouser...  
3  Emma will be home soon. Will will pick her up ...  
4  Jane lost her calendar. Ollie and Jane have lu..

e:\Final Year Project\Speech-to-Text Summarization System for Smart Note-Taking\Virtual\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



🔹 BERTScore:
Precision: 0.9184418320655823
Recall: 0.9132579565048218
F1: 0.9156225323677063


100%|██████████| 819/819 [00:00<00:00, 2239.33it/s]

🔹 ROUGE Scores:
ROUGE-1: 0.5017670180785665
ROUGE-2: 0.25574369499853145
ROUGE-L: 0.41718074050712994
✅ Results saved


### **3. PEGASUS**

In [8]:
file_path = "../experiments/summarization/pegasus_samsum/predictions.csv"

df_bart_english = pd.read_csv(file_path)

print(df_bart_english.head())

P, R, F1 = evaluate_bertscore_english(df_bart_english)
rouge1, rouge2, rougel = evaluate_rouge_english(df_bart_english)
output_dir = "../evaluation/summarization/pegasus_samsum"
rouge_results = (rouge1, rouge2, rougel)
bertscore_results = (P, R, F1)
save_results(df_bart_english, rouge_results, bertscore_results)

e:\Final Year Project\Speech-to-Text Summarization System for Smart Note-Taking\Virtual\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


                                            dialogue  \
0  Hannah: Hey, do you have Betty's number?\nAman...   
1  Eric: MACHINE!\nRob: That's so gr8!\nEric: I k...   
2  Lenny: Babe, can you help me with something?\n...   
3  Will: hey babe, what do you want for dinner to...   
4  Ollie: Hi , are you in Warsaw\nJane: yes, just...   

                                   reference_summary  \
0  Hannah needs Betty's number but Amanda doesn't...   
1  Eric and Rob are going to watch a stand-up on ...   
2  Lenny can't decide which trousers to buy. Bob ...   
3  Emma will be home soon and she will let Will k...   
4  Jane is in Warsaw. Ollie and Jane has a party....   

                                   generated_summary  
0  In this week's episode of Game of Thrones, Han...  
1  In this week's episode of The Big Bang Theory,...  
2  Bob: I've got two pairs of trousers, one in pu...  
3  In this week's episode of The One Show, Will a...  
4  In this week's episode of The One Show, Jane m..

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



🔹 BERTScore:
Precision: 0.8312391638755798
Recall: 0.8680895566940308
F1: 0.8488648533821106


100%|██████████| 819/819 [00:00<00:00, 1517.78it/s]

🔹 ROUGE Scores:
ROUGE-1: 0.20943978305792033
ROUGE-2: 0.042016650794853755
ROUGE-L: 0.15819658482427779
✅ Results saved


### **4. IndicBART**

In [9]:
file_path = "../experiments/summarization/IndicBART_samsum/predictions.csv"

df_bart_english = pd.read_csv(file_path)

print(df_bart_english.head())

P, R, F1 = evaluate_bertscore_english(df_bart_english)
rouge1, rouge2, rougel = evaluate_rouge_english(df_bart_english)
output_dir = "../evaluation/summarization/IndicBART_samsum"
rouge_results = (rouge1, rouge2, rougel)
bertscore_results = (P, R, F1)
save_results(df_bart_english, rouge_results, bertscore_results)

e:\Final Year Project\Speech-to-Text Summarization System for Smart Note-Taking\Virtual\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


                                            dialogue  \
0  Hannah: Hey, do you have Betty's number?\nAman...   
1  Eric: MACHINE!\nRob: That's so gr8!\nEric: I k...   
2  Lenny: Babe, can you help me with something?\n...   
3  Will: hey babe, what do you want for dinner to...   
4  Ollie: Hi , are you in Warsaw\nJane: yes, just...   

                                   reference_summary  \
0  Hannah needs Betty's number but Amanda doesn't...   
1  Eric and Rob are going to watch a stand-up on ...   
2  Lenny can't decide which trousers to buy. Bob ...   
3  Emma will be home soon and she will let Will k...   
4  Jane is in Warsaw. Ollie and Jane has a party....   

                                   generated_summary  
0  नॆंबर् Hannah: Hey, do you have Betty's number...  
1  नॆंबर् Eric: MACHINE! Rob: That's so gr8! Eric...  
2  नॆंबर् Lenny: Babe, can you help me with somet...  
3  Will: hey babe, what do you want for dinner to...  
4  नॆंबर् Ollie: Hi , are you in Warsaw Jane: yes..

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



🔹 BERTScore:
Precision: 0.7909432053565979
Recall: 0.8842251300811768
F1: 0.8345638513565063


100%|██████████| 819/819 [00:00<00:00, 1062.61it/s]

🔹 ROUGE Scores:
ROUGE-1: 0.263647162268862
ROUGE-2: 0.07962555077189942
ROUGE-L: 0.20000635904421668
✅ Results saved


### ***Final Observations:***

- Even though all the models have very good scores in summarization, they lack contextual awareness as well as struggle with noise, and can't work directly for summarization.
- We need to either make them contextually aware or add a context correction model in between like an LLM(Gamma 2b).

### **5. Gamma 2B + BART**

In [10]:
file_path = "../experiments/summarization/Gamma_bart_samsum/predictions.csv"

df_bart_english = pd.read_csv(file_path)

print(df_bart_english.head())

P, R, F1 = evaluate_bertscore_english(df_bart_english)
rouge1, rouge2, rougel = evaluate_rouge_english(df_bart_english)
output_dir = "../evaluation/summarization/Gamma_bart_samsum"
rouge_results = (rouge1, rouge2, rougel)
bertscore_results = (P, R, F1)
save_results(df_bart_english, rouge_results, bertscore_results)

e:\Final Year Project\Speech-to-Text Summarization System for Smart Note-Taking\Virtual\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


                                            dialogue  \
0  Hannah: Hey, do you have Betty's number?\nAman...   
1  Eric: MACHINE!\nRob: That's so gr8!\nEric: I k...   
2  Lenny: Babe, can you help me with something?\n...   
3  Will: hey babe, what do you want for dinner to...   
4  Ollie: Hi , are you in Warsaw\nJane: yes, just...   

                                   reference_summary  \
0  Hannah needs Betty's number but Amanda doesn't...   
1  Eric and Rob are going to watch a stand-up on ...   
2  Lenny can't decide which trousers to buy. Bob ...   
3  Emma will be home soon and she will let Will k...   
4  Jane is in Warsaw. Ollie and Jane has a party....   

                                   generated_summary  
0  the last time they were at the park together, ...  
1  the train part of the stand up stand up is his...  
2  lenny will buy the first pair of purple trouse...  
3  emma will pick up will and will tell her when ...  
4  jane is on her way to Morocco, but time consum..

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



🔹 BERTScore:
Precision: 0.8690388798713684
Recall: 0.8791248202323914
F1: 0.8738802075386047


100%|██████████| 819/819 [00:00<00:00, 1808.83it/s]

🔹 ROUGE Scores:
ROUGE-1: 0.28098473896777343
ROUGE-2: 0.0983459707305061
ROUGE-L: 0.2187408710059978
✅ Results saved


***Observation:***

- The Gamma 2B + BART has good metrics, however, the predictions are still of context.


***Final Conclusion:***

- Instead of just a summarizer model we will need a model that understands the context, speaker relationship, noisy ASR structure and long range dialogue flow of the conversation as well.
- Good Metrics ≠ perfect contextual quality.

---
---

## **HINDI**

**ROUGE EVALUATION**

In [3]:
def evaluate_rouge_hindi(df):

    scorer = rouge_scorer.RougeScorer(
        ['rouge1', 'rouge2', 'rougeL'],
        use_stemmer=False
    )

    rouge1_scores = []
    rouge2_scores = []
    rougel_scores = []

    for _, row in tqdm(df.iterrows(), total=len(df)):

        reference = str(row["reference_summary"])
        prediction = str(row["generated_summary"])

        scores = scorer.score(reference, prediction)

        rouge1_scores.append(scores["rouge1"].fmeasure)
        rouge2_scores.append(scores["rouge2"].fmeasure)
        rougel_scores.append(scores["rougeL"].fmeasure)

    rouge1 = sum(rouge1_scores) / len(rouge1_scores)
    rouge2 = sum(rouge2_scores) / len(rouge2_scores)
    rougel = sum(rougel_scores) / len(rougel_scores)

    return rouge1, rouge2, rougel

**BERTSCORE EVALUATION**

In [4]:
def evaluate_bertscore_hindi(df):

    predictions = (
        df["generated_summary"]
        .fillna("")
        .astype(str)
        .tolist()
    )

    references = (
        df["reference_summary"]
        .fillna("")
        .astype(str)
        .tolist()
    )

    P, R, F1 = score(
        predictions,
        references,
        lang="hi",
        batch_size=2,
        verbose=True
    )

    return (
        P.mean().item(),
        R.mean().item(),
        F1.mean().item()
    )

**SAVE RESULTS**

In [5]:
def save_results(
    output_dir,
    rouge_results,
    bertscore_results
):

    os.makedirs(output_dir, exist_ok=True)

    rouge1, rouge2, rougel = rouge_results
    P, R, F1 = bertscore_results

    results = {
        "ROUGE-1": rouge1,
        "ROUGE-2": rouge2,
        "ROUGE-L": rougel,
        "Precision": P,
        "Recall": R,
        "BERTScore-F1": F1
    }

    # Save JSON
    with open(
        os.path.join(output_dir, "results.json"),
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(results, f, indent=4, ensure_ascii=False)

    print("\n✅ Results Saved")
    print(json.dumps(results, indent=4, ensure_ascii=False))

### **1. Flan-T5(finetuned)**

In [14]:
# ------------------------------------------
# LOAD CSV
# ------------------------------------------
file_path = "../experiments/summarization/flant5_hindi/predictions.csv"

df_flan_hindi = pd.read_csv(file_path)

print(df_flan_hindi.head())

# ------------------------------------------
# EVALUATION
# ------------------------------------------
P, R, F1 = evaluate_bertscore_hindi(df_flan_hindi)

rouge1, rouge2, rougel = evaluate_rouge_hindi(df_flan_hindi)

# ------------------------------------------
# SAVE RESULTS
# ------------------------------------------
output_dir = "../evaluation/summarization/flant5_hindi"

rouge_results = (rouge1, rouge2, rougel)

bertscore_results = (P, R, F1)

save_results(
    output_dir,
    rouge_results,
    bertscore_results
)

                                            dialogue  \
0  प्रधानमंत्री नरेंद्र मोदी पठानकोट एयरबेस पहुंच...   
1  सचिन तेंदुलकर ने एकदिवसीय अंतरराष्ट्रीय क्रिके...   
2  केंद्रीय गृह राज्य मंत्री आर. पी. एन. सिंह ने ...   
3  भारतीय जनता पार्टी (बीजेपी) के राष्ट्रीय अध्यक...   
4  ऋषभ पंत की कभी कभार इस बात के लिए आलोचना की जा...   

                                   reference_summary generated_summary  
0  पठानकोट पहुंचे PM मोदी, एयरबेस का जायजा ले बॉर...               NaN  
1  सचिन ने देशवासियों को समर्पित किया अपना दोहरा शतक               NaN  
2  एनआईए करेगी छत्तीसगढ़ में सुरक्षा खामियों की ज...               NaN  
3  सीधी बात:  शाह बोले- हमारा बस चलता तो अब तक मं...               NaN  
4  ऋषभ पंत के पास यूनिक टैलेंट, उसके साथ छेड़छाड़ न...               NaN  


e:\Final Year Project\Speech-to-Text Summarization System for Smart Note-Taking\Virtual\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


calculating scores...
computing bert embedding.


100%|██████████| 1052/1052 [01:38<00:00, 10.71it/s]


computing greedy matching.


100%|██████████| 1042/1042 [00:11<00:00, 87.30it/s]


done in 110.29 seconds, 604.34 sentences/sec


100%|██████████| 66653/66653 [00:03<00:00, 17752.77it/s]


✅ Results Saved
{
    "ROUGE-1": 0.0030085274743088196,
    "ROUGE-2": 0.0004136984352714693,
    "ROUGE-L": 0.002989719905229307,
    "Precision": 0.6729724407196045,
    "Recall": 0.5652748942375183,
    "BERTScore-F1": 0.6141868829727173
}


### **2. IndicBART(Finetuned) Hindi**

In [15]:
# ------------------------------------------
# LOAD CSV
# ------------------------------------------
file_path = "../experiments/summarization/indicbart_hindi/predictions.csv"

df_indicbart_hindi = pd.read_csv(file_path)

print(df_indicbart_hindi.head())

# ------------------------------------------
# EVALUATION
# ------------------------------------------
P, R, F1 = evaluate_bertscore_hindi(df_indicbart_hindi)

rouge1, rouge2, rougel = evaluate_rouge_hindi(df_indicbart_hindi)

# ------------------------------------------
# SAVE RESULTS
# ------------------------------------------
output_dir = "../evaluation/summarization/indicbart_hindi"

rouge_results = (rouge1, rouge2, rougel)

bertscore_results = (P, R, F1)

save_results(
    output_dir,
    rouge_results,
    bertscore_results
)

                                            dialogue  \
0  प्रधानमंत्री नरेंद्र मोदी पठानकोट एयरबेस पहुंच...   
1  सचिन तेंदुलकर ने एकदिवसीय अंतरराष्ट्रीय क्रिके...   
2  केंद्रीय गृह राज्य मंत्री आर. पी. एन. सिंह ने ...   
3  भारतीय जनता पार्टी (बीजेपी) के राष्ट्रीय अध्यक...   
4  ऋषभ पंत की कभी कभार इस बात के लिए आलोचना की जा...   

                                   reference_summary generated_summary  
0  पठानकोट पहुंचे PM मोदी, एयरबेस का जायजा ले बॉर...               NaN  
1  सचिन ने देशवासियों को समर्पित किया अपना दोहरा शतक               NaN  
2  एनआईए करेगी छत्तीसगढ़ में सुरक्षा खामियों की ज...               NaN  
3  सीधी बात:  शाह बोले- हमारा बस चलता तो अब तक मं...               NaN  
4  ऋषभ पंत के पास यूनिक टैलेंट, उसके साथ छेड़छाड़ न...               NaN  


e:\Final Year Project\Speech-to-Text Summarization System for Smart Note-Taking\Virtual\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


calculating scores...
computing bert embedding.


100%|██████████| 1038/1038 [01:28<00:00, 11.71it/s]


computing greedy matching.


100%|██████████| 1042/1042 [00:12<00:00, 84.98it/s]


done in 101.05 seconds, 659.63 sentences/sec


100%|██████████| 66653/66653 [00:03<00:00, 19478.19it/s]


✅ Results Saved
{
    "ROUGE-1": 6.0012302522017014e-05,
    "ROUGE-2": 0.0,
    "ROUGE-L": 6.0012302522017014e-05,
    "Precision": 0.6742691397666931,
    "Recall": 0.5657155513763428,
    "BERTScore-F1": 0.6150322556495667
}


### ***Critical findings:***

- Finetuning multilingual seq2seq models for Hindi summarization is significantly more unstable than than English summarization on Limited Hardware.
- Need to create new better summarizer from scratch or,
- Use of prebuild models with MuRIL.

### **3. IndicBart(pretrained)**

In [7]:
def evaluate_bertscore_hindi_IndicBART(df, chunk_size=100):

    import gc
    import torch

    predictions = (
        df["generated_summary"]
        .fillna("")
        .astype(str)
        .tolist()
    )

    references = (
        df["reference_summary"]
        .fillna("")
        .astype(str)
        .tolist()
    )

    precision_scores = []
    recall_scores = []
    f1_scores = []

    total_chunks = len(predictions) // chunk_size + 1

    for i in range(0, len(predictions), chunk_size):

        print(f"Processing chunk {i//chunk_size + 1}/{total_chunks}")

        pred_chunk = predictions[i:i+chunk_size]
        ref_chunk = references[i:i+chunk_size]

        P, R, F1 = score(
            pred_chunk,
            ref_chunk,
            lang="hi",
            batch_size=2,
            verbose=False
        )

        precision_scores.extend(P.tolist())
        recall_scores.extend(R.tolist())
        f1_scores.extend(F1.tolist())

        # MEMORY CLEANUP
        gc.collect()
        torch.cuda.empty_cache()

    return (
        sum(precision_scores) / len(precision_scores),
        sum(recall_scores) / len(recall_scores),
        sum(f1_scores) / len(f1_scores)
    )

In [8]:
import re

# ------------------------------------------
# LOAD CSV
# ------------------------------------------
file_path = "../experiments/summarization/indicbart_hindi_pretrained/predictions.csv"

df_indicbart_hindi_pretrained = pd.read_csv(file_path)

# ------------------------------------------
# CLEAN DATA
# ------------------------------------------
df_indicbart_hindi_pretrained = df_indicbart_hindi_pretrained.fillna("")

df_indicbart_hindi_pretrained["generated_summary"] = (
    df_indicbart_hindi_pretrained["generated_summary"]
    .astype(str)
    .apply(
        lambda x: re.sub(
            r"[^\u0900-\u097F\s.,!?a-zA-Z0-9]",
            " ",
            x
        )
    )
    .str.strip()
)

# Remove empty predictions
df_indicbart_hindi_pretrained = (
    df_indicbart_hindi_pretrained[
        df_indicbart_hindi_pretrained["generated_summary"] != ""
    ]
)

print(df_indicbart_hindi_pretrained.head())

# ------------------------------------------
# EVALUATION
# ------------------------------------------
P, R, F1 = evaluate_bertscore_hindi_IndicBART(
    df_indicbart_hindi_pretrained
)

rouge1, rouge2, rougel = evaluate_rouge_hindi(
    df_indicbart_hindi_pretrained
)

# ------------------------------------------
# SAVE RESULTS
# ------------------------------------------
output_dir = "../evaluation/summarization/indicbart_hindi_pretrained"

rouge_results = (rouge1, rouge2, rougel)

bertscore_results = (P, R, F1)

save_results(
    output_dir,
    rouge_results,
    bertscore_results
)

                                            dialogue  \
0  प्रधानमंत्री नरेंद्र मोदी पठानकोट एयरबेस पहुंच...   
1  सचिन तेंदुलकर ने एकदिवसीय अंतरराष्ट्रीय क्रिके...   
2  केंद्रीय गृह राज्य मंत्री आर. पी. एन. सिंह ने ...   
3  भारतीय जनता पार्टी (बीजेपी) के राष्ट्रीय अध्यक...   
4  ऋषभ पंत की कभी कभार इस बात के लिए आलोचना की जा...   

                                   reference_summary  \
0  पठानकोट पहुंचे PM मोदी, एयरबेस का जायजा ले बॉर...   
1  सचिन ने देशवासियों को समर्पित किया अपना दोहरा शतक   
2  एनआईए करेगी छत्तीसगढ़ में सुरक्षा खामियों की ज...   
3  सीधी बात:  शाह बोले- हमारा बस चलता तो अब तक मं...   
4  ऋषभ पंत के पास यूनिक टैलेंट, उसके साथ छेड़छाड़ न...   

                                   generated_summary  
0  प्रधानमंत्री नरेंद्र मोदी पठानकोट एयरबेस पहुंच...  
1  सचिन तेंदुलकर ने एकदिवसीय अंतरराष्ट्रीय क्रिके...  
2  केंद्रीय गृह राज्य मंत्री आर. पी. एन. सिंह ने ...  
3  भारतीय जनता पार्टी  बीजेपी  के राष्ट्रीय अध्यक...  
4  ऋषभ पंत की कभी कभार इस बात के लिए आलोचना की जा..

e:\Final Year Project\Speech-to-Text Summarization System for Smart Note-Taking\Virtual\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Processing chunk 2/667
Processing chunk 3/667
Processing chunk 4/667
Processing chunk 5/667
Processing chunk 6/667
Processing chunk 7/667
Processing chunk 8/667
Processing chunk 9/667
Processing chunk 10/667
Processing chunk 11/667
Processing chunk 12/667
Processing chunk 13/667
Processing chunk 14/667
Processing chunk 15/667
Processing chunk 16/667
Processing chunk 17/667
Processing chunk 18/667
Processing chunk 19/667
Processing chunk 20/667
Processing chunk 21/667
Processing chunk 22/667
Processing chunk 23/667
Processing chunk 24/667
Processing chunk 25/667
Processing chunk 26/667
Processing chunk 27/667
Processing chunk 28/667
Processing chunk 29/667
Processing chunk 30/667
Processing chunk 31/667
Processing chunk 32/667
Processing chunk 33/667
Processing chunk 34/667
Processing chunk 35/667
Processing chunk 36/667
Processing chunk 37/667
Processing chunk 38/667
Processing chunk 39/667
Processing chunk 40/667
Processing chunk 41/667
Processing chunk 42/667
Processing chunk 43/667


Processing chunk 103/667
Processing chunk 104/667
Processing chunk 105/667
Processing chunk 106/667
Processing chunk 107/667
Processing chunk 108/667
Processing chunk 109/667
Processing chunk 110/667
Processing chunk 111/667
Processing chunk 112/667
Processing chunk 113/667
Processing chunk 114/667
Processing chunk 115/667
Processing chunk 116/667
Processing chunk 117/667
Processing chunk 118/667
Processing chunk 119/667
Processing chunk 120/667
Processing chunk 121/667
Processing chunk 122/667
Processing chunk 123/667
Processing chunk 124/667
Processing chunk 125/667
Processing chunk 126/667
Processing chunk 127/667
Processing chunk 128/667
Processing chunk 129/667
Processing chunk 130/667
Processing chunk 131/667
Processing chunk 132/667
Processing chunk 133/667
Processing chunk 134/667
Processing chunk 135/667
Processing chunk 136/667
Processing chunk 137/667
Processing chunk 138/667
Processing chunk 139/667
Processing chunk 140/667
Processing chunk 141/667
Processing chunk 142/667


Processing chunk 304/667
Processing chunk 305/667
Processing chunk 306/667
Processing chunk 307/667
Processing chunk 308/667
Processing chunk 309/667
Processing chunk 310/667
Processing chunk 311/667
Processing chunk 312/667
Processing chunk 313/667
Processing chunk 314/667
Processing chunk 315/667
Processing chunk 316/667
Processing chunk 317/667
Processing chunk 318/667
Processing chunk 319/667
Processing chunk 320/667
Processing chunk 321/667
Processing chunk 322/667
Processing chunk 323/667
Processing chunk 324/667
Processing chunk 325/667
Processing chunk 326/667
Processing chunk 327/667
Processing chunk 328/667
Processing chunk 329/667
Processing chunk 330/667
Processing chunk 331/667
Processing chunk 332/667
Processing chunk 333/667
Processing chunk 334/667
Processing chunk 335/667
Processing chunk 336/667
Processing chunk 337/667
Processing chunk 338/667
Processing chunk 339/667
Processing chunk 340/667
Processing chunk 341/667
Processing chunk 342/667
Processing chunk 343/667


Processing chunk 519/667
Processing chunk 520/667
Processing chunk 521/667
Processing chunk 522/667
Processing chunk 523/667
Processing chunk 524/667
Processing chunk 525/667
Processing chunk 526/667
Processing chunk 527/667
Processing chunk 528/667
Processing chunk 529/667
Processing chunk 530/667
Processing chunk 531/667
Processing chunk 532/667
Processing chunk 533/667
Processing chunk 534/667
Processing chunk 535/667
Processing chunk 536/667
Processing chunk 537/667
Processing chunk 538/667
Processing chunk 539/667
Processing chunk 540/667
Processing chunk 541/667
Processing chunk 542/667
Processing chunk 543/667
Processing chunk 544/667
Processing chunk 545/667
Processing chunk 546/667
Processing chunk 547/667
Processing chunk 548/667
Processing chunk 549/667
Processing chunk 550/667
Processing chunk 551/667
Processing chunk 552/667
Processing chunk 553/667
Processing chunk 554/667
Processing chunk 555/667
Processing chunk 556/667
Processing chunk 557/667
Processing chunk 558/667


Processing chunk 576/667
Processing chunk 577/667
Processing chunk 578/667
Processing chunk 579/667
Processing chunk 580/667
Processing chunk 581/667
Processing chunk 582/667
Processing chunk 583/667
Processing chunk 584/667
Processing chunk 585/667
Processing chunk 586/667
Processing chunk 587/667
Processing chunk 588/667
Processing chunk 589/667
Processing chunk 590/667
Processing chunk 591/667
Processing chunk 592/667
Processing chunk 593/667
Processing chunk 594/667
Processing chunk 595/667
Processing chunk 596/667
Processing chunk 597/667
Processing chunk 598/667
Processing chunk 599/667
Processing chunk 600/667
Processing chunk 601/667
Processing chunk 602/667
Processing chunk 603/667
Processing chunk 604/667
Processing chunk 605/667
Processing chunk 606/667
Processing chunk 607/667
Processing chunk 608/667
Processing chunk 609/667
Processing chunk 610/667
Processing chunk 611/667
Processing chunk 612/667
Processing chunk 613/667
Processing chunk 614/667
Processing chunk 615/667


100%|██████████| 66653/66653 [00:03<00:00, 19241.66it/s]


✅ Results Saved
{
    "ROUGE-1": 0.11841067834130943,
    "ROUGE-2": 0.02314232980939239,
    "ROUGE-L": 0.11664512600588707,
    "Precision": 0.6672090205744112,
    "Recall": 0.7529847613815792,
    "BERTScore-F1": 0.7070239222191184
}


### **Flan-T5**

### **4. Flan-T5(pretrained)**

In [16]:
# ------------------------------------------
# LOAD CSV
# ------------------------------------------
file_path = "../experiments/summarization/flant5_hindi_pretrained/predictions.csv"

df_flant5_hindi_pretrained = pd.read_csv(file_path)

print(df_flant5_hindi_pretrained.head())

# ------------------------------------------
# EVALUATION
# ------------------------------------------
P, R, F1 = evaluate_bertscore_hindi(df_flant5_hindi_pretrained)

rouge1, rouge2, rougel = evaluate_rouge_hindi(df_flant5_hindi_pretrained)

# ------------------------------------------
# SAVE RESULTS
# ------------------------------------------
output_dir = "../evaluation/summarization/flant5_hindi_pretrained"

rouge_results = (rouge1, rouge2, rougel)

bertscore_results = (P, R, F1)

save_results(
    output_dir,
    rouge_results,
    bertscore_results
)

                                            dialogue  \
0  प्रधानमंत्री नरेंद्र मोदी पठानकोट एयरबेस पहुंच...   
1  सचिन तेंदुलकर ने एकदिवसीय अंतरराष्ट्रीय क्रिके...   
2  केंद्रीय गृह राज्य मंत्री आर. पी. एन. सिंह ने ...   
3  भारतीय जनता पार्टी (बीजेपी) के राष्ट्रीय अध्यक...   
4  ऋषभ पंत की कभी कभार इस बात के लिए आलोचना की जा...   

                                   reference_summary generated_summary  
0  पठानकोट पहुंचे PM मोदी, एयरबेस का जायजा ले बॉर...               NaN  
1  सचिन ने देशवासियों को समर्पित किया अपना दोहरा शतक               NaN  
2  एनआईए करेगी छत्तीसगढ़ में सुरक्षा खामियों की ज...               NaN  
3  सीधी बात:  शाह बोले- हमारा बस चलता तो अब तक मं...               NaN  
4  ऋषभ पंत के पास यूनिक टैलेंट, उसके साथ छेड़छाड़ न...               NaN  


e:\Final Year Project\Speech-to-Text Summarization System for Smart Note-Taking\Virtual\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


calculating scores...
computing bert embedding.


100%|██████████| 1049/1049 [01:30<00:00, 11.63it/s]


computing greedy matching.


100%|██████████| 1042/1042 [00:11<00:00, 90.72it/s]


done in 101.81 seconds, 654.70 sentences/sec


100%|██████████| 66653/66653 [00:03<00:00, 19330.47it/s]


✅ Results Saved
{
    "ROUGE-1": 0.002807657185361443,
    "ROUGE-2": 0.001482007171506759,
    "ROUGE-L": 0.002805349019879827,
    "Precision": 0.6737853288650513,
    "Recall": 0.5658924579620361,
    "BERTScore-F1": 0.6148905158042908
}


# ***FINAL CONCLUSION:***

- Hindi summarization proved significantly more challenging than English summarization due to the complexity of multilingual text generation, noisy ASR outputs, and the morphologically rich nature of Hindi.
- Fine-tuning multilingual models such as IndicBART and FLAN-T5 on limited hardware resulted in unstable training behavior, including empty or degraded outputs, indicating that multilingual sequence-to-sequence models are highly sensitive to preprocessing quality, tokenization strategy, and training stability.
- The pretrained IndicBART model, despite not being explicitly fine-tuned for summarization, was able to preserve the core semantic meaning and generate contextually relevant extractive summaries from long Hindi news articles.
- However, the generated outputs were often verbose and extractive rather than concise abstractive summaries or headlines, showing that pretrained multilingual models require task-specific adaptation for high-quality Hindi summarization.
- Evaluation using ROUGE and BERTScore further demonstrated that semantic understanding was preserved even when lexical overlap was lower, reinforcing the observation that multilingual summarization models can retain contextual meaning despite imperfect generation quality.
- Overall, the experiments suggest that pretrained multilingual transformer models are capable of understanding Hindi context, but robust Hindi summarization still requires:
    - better transcript preprocessing,
    - context-aware formatting,
    - cleaner tokenization,
    - and larger-scale stable fine-tuning setups.
- Therefore, for the final system:
    - English summarization uses the fine-tuned BART model,
    - while Hindi summarization relies on pretrained multilingual models combined with transcript preprocessing and contextual correction strategies, that is, IndicBART pre-trained model.